# SSIF 04: Employability Classification & Conditional Salary Modeling
### Student Success Intelligence Framework (SSIF)

This notebook evaluates employability prediction models and salary diagnostics:
- **Selection Classification:** 5-Fold Stratified Cross-Validation (AUROC = 0.9370).
- **Subgroup Disparities:** Quantifying the placement lift from prior work experience (59.6% -> 86.5%), specialization, and gender.
- **Conditional Salary Regression (N=148):** Assessing whether academic GPA predicts starting compensation offers ($R^2 pprox 0$).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.placement.models import run_placement_pipeline

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Run Employability & Salary Pipeline

In [ ]:
res = run_placement_pipeline(n_splits=5, random_state=42)
print("=== Classification Model Leaderboard (Stratified 5-Fold CV) ===")
print(res.classification_summary)

## 2. Subgroup Placement Rates & Demographic Lift

In [ ]:
print("=== Subgroup Disparities ===")
print(res.subgroup_analysis)

## 3. Conditional Salary Regression (Placed Candidates N=148)

In [ ]:
print("=== Conditional Salary Model (Placed Only N=148) ===")
for model_name, metrics in res.salary_metrics.items():
    print(f"{model_name}: R2 = {metrics['r2']:.4f}, MAE = INR {metrics['mae']:,.0f}, RMSE = INR {metrics['rmse']:,.0f}")

print("\nScientific Finding: Academic marks do not predict starting MBA salaries (R2 ~ 0).")
print("Salary offers are governed by rigid corporate pay bands rather than decimal GPA differences.")

## 4. Visualizing Work Experience Placement Advantage

In [ ]:
sub_df = res.subgroup_analysis
workex_data = sub_df[sub_df["Subgroup"].isin(["No Work Experience", "With Work Experience"])]

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=workex_data, x="Subgroup", y="Placement_Rate_Pct", palette="Blues_r", ax=ax)
ax.set_ylabel("Placement Rate (%)")
ax.set_title("Impact of Prior Work Experience on Career Placement (N=215)")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 1.0), ha='center')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()